# 00 Common Setup

This notebook prepares the shared experimental environment for the thesis project.

It performs:
- reproducibility setup
- dataset path validation
- transform and dataloader preparation
- class alignment verification between PlantVillage and PlantDoc
- export of shared configuration files

Outputs:
- shared configuration JSON
- dataset summary CSV
- class mapping JSON
- ZIP archive of setup outputs

In [1]:
# ----------------------------------------
# Section 1: Imports
# ----------------------------------------

import os
import json
import random
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
# ----------------------------------------
# Section 2: Reproducibility setup
# ----------------------------------------

SEED = 42

def seed_everything(seed: int = 42) -> None:
    """
    Set random seeds for reproducibility across Python, NumPy, and PyTorch.
    """
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id: int) -> None:
    """
    Ensure each DataLoader worker uses a deterministic seed.
    """
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

seed_everything(SEED)

print("Reproducibility setup completed")
print(f"Global seed: {SEED}")

Reproducibility setup completed
Global seed: 42


In [3]:
# ----------------------------------------
# Section 3: Configuration
# ----------------------------------------

CONFIG = {
    "seed": SEED,
    "image_size": 224,
    "batch_size": 32,
    "num_workers": 2,
    "plantvillage_root": "/kaggle/input/datasets/thedataeng/plantvillage",
    "plantdoc_root": "/kaggle/input/datasets/thedataeng/plantdoc",
    "output_root": "/kaggle/working/thesis_outputs/common_setup",
}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_ROOT = Path(CONFIG["output_root"])
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Configuration loaded")
print(f"Device: {DEVICE}")
print(f"Output root: {OUTPUT_ROOT}")

Configuration loaded
Device: cuda
Output root: /kaggle/working/thesis_outputs/common_setup


In [4]:
# ----------------------------------------
# Section 4: Helper functions
# ----------------------------------------

def ensure_dir(path: Path) -> Path:
    """
    Create a directory if it does not exist and return the Path object.
    """
    path.mkdir(parents=True, exist_ok=True)
    return path

def get_eval_transform(image_size: int = 224):
    """
    Create the standard evaluation transform used across notebooks.
    """
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
    ])

def get_train_transform(image_size: int = 224):
    """
    Create the standard training transform used across notebooks.
    """
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
    ])

In [5]:
# ----------------------------------------
# Section 5: Dataset path validation
# ----------------------------------------

pv_root = Path(CONFIG["plantvillage_root"])
pd_root = Path(CONFIG["plantdoc_root"])

train_dir = pv_root / "train"
val_dir = pv_root / "val"
test_dir = pv_root / "test"

assert train_dir.exists(), f"Missing PlantVillage train directory: {train_dir}"
assert val_dir.exists(), f"Missing PlantVillage val directory: {val_dir}"
assert test_dir.exists(), f"Missing PlantVillage test directory: {test_dir}"
assert pd_root.exists(), f"Missing PlantDoc directory: {pd_root}"

print("Dataset paths validated successfully")
print(f"PlantVillage train: {train_dir}")
print(f"PlantVillage val:   {val_dir}")
print(f"PlantVillage test:  {test_dir}")
print(f"PlantDoc root:      {pd_root}")

Dataset paths validated successfully
PlantVillage train: /kaggle/input/datasets/thedataeng/plantvillage/train
PlantVillage val:   /kaggle/input/datasets/thedataeng/plantvillage/val
PlantVillage test:  /kaggle/input/datasets/thedataeng/plantvillage/test
PlantDoc root:      /kaggle/input/datasets/thedataeng/plantdoc


In [6]:
# ----------------------------------------
# Section 6: Dataset loading
# ----------------------------------------

train_tfms = get_train_transform(CONFIG["image_size"])
eval_tfms = get_eval_transform(CONFIG["image_size"])

train_dataset = datasets.ImageFolder(train_dir, transform=train_tfms)
val_dataset = datasets.ImageFolder(val_dir, transform=eval_tfms)
test_dataset = datasets.ImageFolder(test_dir, transform=eval_tfms)
plantdoc_dataset = datasets.ImageFolder(pd_root, transform=eval_tfms)

print("Datasets loaded successfully")
print(f"Train samples:      {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples:       {len(test_dataset)}")
print(f"PlantDoc samples:   {len(plantdoc_dataset)}")

Datasets loaded successfully
Train samples:      25795
Validation samples: 5518
Test samples:       5553
PlantDoc samples:   2555


In [7]:
# ----------------------------------------
# Section 7: Class alignment verification
# ----------------------------------------

pv_classes = train_dataset.classes
pd_classes = plantdoc_dataset.classes

classes_match = pv_classes == pd_classes

print("Class alignment check completed")
print(f"PlantVillage classes: {len(pv_classes)}")
print(f"PlantDoc classes:     {len(pd_classes)}")
print(f"Class lists match:    {classes_match}")

if not classes_match:
    raise ValueError("PlantVillage and PlantDoc class order does not match")

print("Class alignment is valid for cross-dataset evaluation")

Class alignment check completed
PlantVillage classes: 27
PlantDoc classes:     27
Class lists match:    True
Class alignment is valid for cross-dataset evaluation


In [8]:
# ----------------------------------------
# Section 8: DataLoader preparation
# ----------------------------------------

generator = torch.Generator()
generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"],
    worker_init_fn=seed_worker,
    generator=generator,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    worker_init_fn=seed_worker,
    generator=generator,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    worker_init_fn=seed_worker,
    generator=generator,
    pin_memory=True,
)

plantdoc_loader = DataLoader(
    plantdoc_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    worker_init_fn=seed_worker,
    generator=generator,
    pin_memory=True,
)

print("DataLoaders created successfully")
print(f"Train batches:    {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches:     {len(test_loader)}")
print(f"PlantDoc batches: {len(plantdoc_loader)}")

DataLoaders created successfully
Train batches:    807
Validation batches: 173
Test batches:     174
PlantDoc batches: 80


In [9]:
# ----------------------------------------
# Section 9: Save shared metadata
# ----------------------------------------

meta_dir = ensure_dir(OUTPUT_ROOT / "metadata")

config_path = meta_dir / "common_config.json"
with open(config_path, "w") as f:
    json.dump(CONFIG, f, indent=2)

class_map_path = meta_dir / "class_mapping.json"
with open(class_map_path, "w") as f:
    json.dump(
        {
            "plantvillage_classes": train_dataset.classes,
            "plantdoc_classes": plantdoc_dataset.classes,
            "class_to_idx": train_dataset.class_to_idx,
        },
        f,
        indent=2
    )

summary_df = pd.DataFrame([
    {"Dataset": "PlantVillage Train", "Samples": len(train_dataset), "Classes": len(train_dataset.classes)},
    {"Dataset": "PlantVillage Validation", "Samples": len(val_dataset), "Classes": len(val_dataset.classes)},
    {"Dataset": "PlantVillage Test", "Samples": len(test_dataset), "Classes": len(test_dataset.classes)},
    {"Dataset": "PlantDoc", "Samples": len(plantdoc_dataset), "Classes": len(plantdoc_dataset.classes)},
])

summary_path = meta_dir / "dataset_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("Shared metadata saved successfully")
print(f"Config saved to:        {config_path}")
print(f"Class mapping saved to: {class_map_path}")
print(f"Dataset summary saved to: {summary_path}")

Shared metadata saved successfully
Config saved to:        /kaggle/working/thesis_outputs/common_setup/metadata/common_config.json
Class mapping saved to: /kaggle/working/thesis_outputs/common_setup/metadata/class_mapping.json
Dataset summary saved to: /kaggle/working/thesis_outputs/common_setup/metadata/dataset_summary.csv


In [10]:
# ----------------------------------------
# Section 10: Preview saved metadata
# ----------------------------------------

print("Dataset summary preview:")
display(summary_df)

print("First 10 PlantVillage classes:")
display(pd.DataFrame({"Class": train_dataset.classes[:10]}))

Dataset summary preview:


,Dataset,Samples,Classes
0,PlantVillage Train,25795,27
1,PlantVillage Validation,5518,27
2,PlantVillage Test,5553,27
3,PlantDoc,2555,27


First 10 PlantVillage classes:


,Class
0,Apple Scab Leaf
1,Apple leaf
2,Apple rust leaf
3,Bell_pepper leaf
4,Bell_pepper leaf spot
5,Blueberry leaf
6,Cherry leaf
7,Corn Gray leaf spot
8,Corn leaf blight
9,Corn rust leaf


In [11]:
# ----------------------------------------
# Section 11: Create ZIP archive
# ----------------------------------------

zip_path = OUTPUT_ROOT.parent / "00_common_setup_outputs.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in OUTPUT_ROOT.rglob("*"):
        if file_path.is_file():
            zf.write(file_path, arcname=file_path.relative_to(OUTPUT_ROOT))

print("ZIP archive created successfully")
print(f"ZIP file: {zip_path}")
print("00_common_setup notebook completed successfully")

ZIP archive created successfully
ZIP file: /kaggle/working/thesis_outputs/00_common_setup_outputs.zip
00_common_setup notebook completed successfully
